<a href="https://colab.research.google.com/github/Jacksonfvsantos/lol-worlds-predictor/blob/main/notebooks/worlds_predictor_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#CÉLULA 1
import os
import subprocess
import warnings

os.environ['PYTHONWARNINGS'] = 'ignore'
warnings.filterwarnings('ignore')

subprocess.run(
    ['pip', 'install', 'urllib3<2', 'mwclient', 'joblib', 'scikit-learn', 'pandas', 'numpy', '-q'],
    capture_output=True
)

!pip install "urllib3<2" mwclient joblib scikit-learn pandas numpy -q

import glob
import pandas as pd
import numpy as np
import mwclient
import joblib
from google.colab import drive

drive.mount('/content/drive')

if not os.path.exists('/content/lol-worlds-predictor'):
    subprocess.run(
        ['git', 'clone', 'https://github.com/Jacksonfvsantos/lol-worlds-predictor.git'],
        capture_output=True
    )
print("✅ Ambiente configurado, Repositório clonado e Google Drive conectado!")

In [ ]:
#CÉLULA 2
caminho_dados_drive = '/content/drive/MyDrive/lol-worlds-predictor-data/data/*.csv'
caminho_local = '/content/lol-worlds-predictor/data/*.csv'
arquivos_csv = glob.glob(caminho_dados_drive) if glob.glob(caminho_dados_drive) else glob.glob(caminho_local)

if not arquivos_csv:
    raise FileNotFoundError("Nenhum arquivo CSV encontrado! Verifique se os arquivos de 2018 a 2026 estão na pasta 'lol-worlds-predictor-data/data/' do seu Drive.")

lista_dfs = [pd.read_csv(arquivo, low_memory=False) for arquivo in arquivos_csv]
df_lol = pd.concat(lista_dfs, ignore_index=True)

print(f"✅ {len(arquivos_csv)} arquivo(s) CSV carregado(s) com sucesso! Total de linhas: {len(df_lol)}")

In [ ]:
#CÉLULA 3
df_times = df_lol[df_lol['position'] == 'team'].copy()

df_features = df_times.groupby('teamname').agg(
    total_jogos=('gameid', 'count'),
    winrate_temporada=('result', 'mean'),
    golddiffat15=('golddiffat15', 'mean'),
    firstblood=('firstblood', 'mean'),
    firstdragon=('firstdragon', 'mean'),
    firsttower=('firsttower', 'mean'),
    vspm=('vspm', 'mean')
).reset_index()

df_features = df_features[df_features['total_jogos'] >= 15].reset_index(drop=True)

print(f"✅ Perfil gerado para {len(df_features)} equipes!")
display(df_features.head(5))

In [ ]:
#CÉLULA 4
df_blue = df_times[df_times['side'] == 'Blue'][['gameid', 'teamname', 'result']]
df_red = df_times[df_times['side'] == 'Red'][['gameid', 'teamname']]

df_confrontos = pd.merge(df_blue, df_red, on='gameid', suffixes=('_blue', '_red'))

df_features_blue = df_features.rename(columns=lambda x: x + '_blue' if x != 'teamname' else 'teamname_blue')
df_features_red = df_features.rename(columns=lambda x: x + '_red' if x != 'teamname' else 'teamname_red')

df_modelo = pd.merge(df_confrontos, df_features_blue, on='teamname_blue')
df_modelo = pd.merge(df_modelo, df_features_red, on='teamname_red')

df_modelo['diff_winrate'] = df_modelo['winrate_temporada_blue'] - df_modelo['winrate_temporada_red']
df_modelo['diff_golddiffat15'] = df_modelo['golddiffat15_blue'] - df_modelo['golddiffat15_red']
df_modelo['diff_firstblood'] = df_modelo['firstblood_blue'] - df_modelo['firstblood_red']
df_modelo['diff_firstdragon'] = df_modelo['firstdragon_blue'] - df_modelo['firstdragon_red']
df_modelo['diff_firsttower'] = df_modelo['firsttower_blue'] - df_modelo['firsttower_red']
df_modelo['diff_vspm'] = df_modelo['vspm_blue'] - df_modelo['vspm_red']

colunas_finais = [
    'gameid', 'teamname_blue', 'teamname_red', 'result',
    'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
    'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
]

df_treino = df_modelo[colunas_finais].copy()
df_treino.dropna(inplace=True)

print(f"✅ Dataset de treino criado com {len(df_treino)} confrontos!")

In [ ]:
#CÉLULA 5
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

features = [
    'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
    'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
]
X = df_treino[features]
y = df_treino['result']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

modelo_lol = LogisticRegression(max_iter=1000)
modelo_lol.fit(X_train, y_train)

y_pred = modelo_lol.predict(X_test)
acuracia = accuracy_score(y_test, y_pred)

print(f"🏆 Acurácia Geral do Modelo: {acuracia:.2%}\n")

pasta_modelos = '/content/drive/MyDrive/lol-worlds-predictor-data/models'
os.makedirs(pasta_modelos, exist_ok=True)

caminho_modelo = f'{pasta_modelos}/modelo_lol_worlds.pkl'
joblib.dump(modelo_lol, caminho_modelo)
print(f"✅ Modelo salvo em segurança: {caminho_modelo}")

In [ ]:
#CÉLULA 6
alias_times = {
    "AL": "Anyone's Legend",
    "BLG": "Bilibili Gaming",
    "CFO": "CTBC Flying Oyster",
    "DK": "Dplus KIA",
    "G2": "G2 Esports",
    "GEN": "Gen.G",
    "HLE": "Hanwha Life Esports",
    "iG": "Invictus Gaming",
    "MKOI": "Movistar KOI",
    "TL": "Team Liquid",
    "TSW": "Team Secret Whales",
    "TES": "Top Esports",
    "C9": "Cloud9",
    "KC": "Karmine Corp",
    "MVK": "MVK Esports",
    "FUR": "FURIA"
}

def prever_confronto(time_azul, time_vermelho, df_stats, modelo):
    time_azul_corr = alias_times.get(time_azul, time_azul)
    time_vermelho_corr = alias_times.get(time_vermelho, time_vermelho)

    stats_azul = df_stats[df_stats['teamname'] == time_azul_corr]
    stats_vermelho = df_stats[df_stats['teamname'] == time_vermelho_corr]

    if stats_azul.empty or stats_vermelho.empty:
        print(f"❌ Erro: '{time_azul_corr}' ou '{time_vermelho_corr}' sem dados no histórico.")
        return None

    diff_winrate = float(stats_azul['winrate_temporada'].values[0] - stats_vermelho['winrate_temporada'].values[0])
    diff_golddiffat15 = float(stats_azul['golddiffat15'].values[0] - stats_vermelho['golddiffat15'].values[0])
    diff_firstblood = float(stats_azul['firstblood'].values[0] - stats_vermelho['firstblood'].values[0])
    diff_firstdragon = float(stats_azul['firstdragon'].values[0] - stats_vermelho['firstdragon'].values[0])
    diff_firsttower = float(stats_azul['firsttower'].values[0] - stats_vermelho['firsttower'].values[0])
    diff_vspm = float(stats_azul['vspm'].values[0] - stats_vermelho['vspm'].values[0])

    X_novo = pd.DataFrame([[
        diff_winrate, diff_golddiffat15, diff_firstblood,
        diff_firstdragon, diff_firsttower, diff_vspm
    ]], columns=[
        'diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
        'diff_firstdragon', 'diff_firsttower', 'diff_vspm'
    ])

    probabilidades = modelo.predict_proba(X_novo)[0]
    prob_vermelho = probabilidades[0] * 100
    prob_azul = probabilidades[1] * 100

    vencedor = time_azul_corr if prob_azul > prob_vermelho else time_vermelho_corr

    return {
        'Blue Side': time_azul_corr,
        'Red Side': time_vermelho_corr,
        'Chance Blue (%)': round(prob_azul, 2),
        'Chance Red (%)': round(prob_vermelho, 2),
        'Vencedor Previsto': vencedor
    }

print("✅ Função de previsão pronta!")

In [ ]:
#CÉLULA 7
import os
import time
import mwclient
import pandas as pd

torneio_alvo = "2026 Season World Championship"
caminho_cache = '/content/drive/MyDrive/lol-worlds-predictor-data/data/cache_chaveamento_2026.csv'

validade_cache = 86400
usar_cache = False

# 1. Verifica se o cache existe, está na validade E possui dados reais
if os.path.exists(caminho_cache):
    idade_arquivo = time.time() - os.path.getmtime(caminho_cache)
    try:
        df_teste = pd.read_csv(caminho_cache)
        if idade_arquivo < validade_cache and len(df_teste) > 0:
            usar_cache = True
    except:
        pass

if usar_cache:
    print("⚡ Carregando chaveamento do CACHE local (Rápido e sem bloqueios)...")
    df_chaveamento = pd.read_csv(caminho_cache)
    print(f"✅ {len(df_chaveamento)} partidas carregadas do cache!")
    display(df_chaveamento.head())

else:
    print("🌐 Cache vazio/inválido. Buscando dados ao vivo na API da Leaguepedia...")
    site = mwclient.Site(
        'lol.fandom.com',
        path='/',
        clients_useragent='lol-worlds-predictor-tcc/1.0 (Projeto de TCC; Unijorge)'
    )

    max_tentativas = 3
    resposta_partidas = None

    for tentativa in range(max_tentativas):
        try:
            # Correção principal: Uso do operador LIKE para pegar Swiss Stage, Playoffs, etc.
            resposta_partidas = site.api('cargoquery',
                limit='max',
                tables="MatchSchedule",
                fields="Team1, Team2, Tab",
                where=f"OverviewPage LIKE '{torneio_alvo}%' AND Team1 IS NOT NULL AND Team2 IS NOT NULL",
                format="json"
            )
            print("✅ Conexão com a Leaguepedia bem-sucedida!")
            break

        except Exception as e:
            print(f"⚠️ Aviso de rede (Tentativa {tentativa + 1}/{max_tentativas}).")
            if tentativa < max_tentativas - 1:
                print("⏳ Pausando por 15 segundos...")
                time.sleep(15)
            else:
                print("❌ Falha crítica: Servidor indisponível.")

    # Processamento dos dados
    if resposta_partidas:
        lista_partidas = []
        for item in resposta_partidas['cargoquery']:
            dados = item['title']
            lista_partidas.append({
                'Fase_do_Torneio': dados['Tab'],
                'Time_Azul': dados['Team1'],
                'Time_Vermelho': dados['Team2']
            })

        df_chaveamento = pd.DataFrame(lista_partidas)

        # Só atualiza o cache no Drive se realmente encontrou os jogos
        if len(df_chaveamento) > 0:
            df_chaveamento.to_csv(caminho_cache, index=False)
            print(f"💾 Novo cache de confrontos salvo com sucesso em: {caminho_cache}")
        else:
            print(f"⚠️ Nenhuma partida encontrada para '{torneio_alvo}'. O cache não foi sobrescrito.")

        print(f"✅ {len(df_chaveamento)} partidas prontas para simulação!")
        if len(df_chaveamento) > 0:
            display(df_chaveamento.head())

In [ ]:
#CÉLULA 8
resultados_simulacao = []

df_jogos_definidos = df_chaveamento[
    (df_chaveamento['Time_Azul'] != 'TBD') &
    (df_chaveamento['Time_Vermelho'] != 'TBD')
].copy()

print(f"🔮 Executando previsões para {len(df_jogos_definidos)} confrontos definidos...\n")

for index, row in df_jogos_definidos.iterrows():
    res = prever_confronto(row['Time_Azul'], row['Time_Vermelho'], df_features, modelo_lol)
    if res:
        res['Fase'] = row['Fase_do_Torneio']
        resultados_simulacao.append(res)

df_previsoes_finais = pd.DataFrame(resultados_simulacao)

print("🚀 SIMULAÇÃO COMPLETA CONCLUÍDA!")
if not df_previsoes_finais.empty:
    display(df_previsoes_finais)
else:
    print("⚠️ Nenhuma previsão pôde ser gerada. Verifique se os nomes dos times batem com o seu histórico (use o dicionário alias_times na Célula 6 se necessário).")

In [ ]:
# CÉLULA 9:
import random

# 1. FUNÇÕES BASE DE MONTE CARLO
def simular_mapa_mc(time_azul, time_vermelho, df_stats, modelo):
    t_azul = alias_times.get(time_azul, time_azul)
    t_verm = alias_times.get(time_vermelho, time_vermelho)

    s_azul = df_stats[df_stats['teamname'] == t_azul]
    s_verm = df_stats[df_stats['teamname'] == t_verm]

    if s_azul.empty or s_verm.empty: return None

    d_win = float(s_azul['winrate_temporada'].values[0] - s_verm['winrate_temporada'].values[0])
    d_gold = float(s_azul['golddiffat15'].values[0] - s_verm['golddiffat15'].values[0])
    d_fb = float(s_azul['firstblood'].values[0] - s_verm['firstblood'].values[0])
    d_fd = float(s_azul['firstdragon'].values[0] - s_verm['firstdragon'].values[0])
    d_ft = float(s_azul['firsttower'].values[0] - s_verm['firsttower'].values[0])
    d_vspm = float(s_azul['vspm'].values[0] - s_verm['vspm'].values[0])

    X_novo = pd.DataFrame([[d_win, d_gold, d_fb, d_fd, d_ft, d_vspm]],
                          columns=['diff_winrate', 'diff_golddiffat15', 'diff_firstblood',
                                   'diff_firstdragon', 'diff_firsttower', 'diff_vspm'])

    prob_azul = modelo.predict_proba(X_novo)[0][1] * 100
    roleta = random.uniform(0, 100)

    return t_azul if roleta <= prob_azul else t_verm

def simular_serie(time_a, time_b, tipo_serie, df_stats, modelo, time_a_blue_jogo1=True):
    vitorias = {time_a: 0, time_b: 0}
    vit_nec = 2 if tipo_serie == 'BO3' else (3 if tipo_serie == 'BO5' else 1)

    t_azul = time_a if time_a_blue_jogo1 else time_b
    t_verm = time_b if time_a_blue_jogo1 else time_a

    while vitorias[time_a] < vit_nec and vitorias[time_b] < vit_nec:
        vencedor_mapa = simular_mapa_mc(t_azul, t_verm, df_stats, modelo)
        if not vencedor_mapa: return None
        vitorias[vencedor_mapa] += 1

        if vitorias[time_a] == vit_nec or vitorias[time_b] == vit_nec: break

        perdedor = time_a if vencedor_mapa == time_b else time_b
        t_azul = perdedor
        t_verm = time_b if perdedor == time_a else time_a

    vencedor_serie = time_a if vitorias[time_a] > vitorias[time_b] else time_b
    return vencedor_serie, vitorias

def rodar_monte_carlo(time_a, time_b, tipo_serie, df_stats, modelo, n_sims=1000):
    vits = {time_a: 0, time_b: 0}
    placares = {}

    for _ in range(n_sims):
        res = simular_serie(time_a, time_b, tipo_serie, df_stats, modelo, random.choice([True, False]))
        if not res: return None

        vencedor_serie, placar = res
        vits[vencedor_serie] += 1

        str_placar = f"{placar[time_a]}-{placar[time_b]}"
        placares[str_placar] = placares.get(str_placar, 0) + 1

    placar_provavel = max(placares, key=placares.get)
    vencedor_estatistico = time_a if vits[time_a] > vits[time_b] else time_b

    return vencedor_estatistico, placar_provavel

# 2. SIMULADOR DO CHAVEAMENTO DO PLAY-IN (Apenas 1 Vaga)
def simular_grupo_playin_uma_vaga(t1, t2, t3, t4, df_stats, modelo, n_sims=500):
    print("==================================================")
    print(f"🏆 INICIANDO SIMULAÇÃO DO GRUPO (Apenas 1 Vaga)")
    print(f"Equipes: {t1}, {t2}, {t3}, {t4} | Simulações: {n_sims}")
    print("==================================================\n")

    # ROUND 1
    print("⚔️ ROUND 1 (Melhor de 3)")
    venc1, plac1 = rodar_monte_carlo(t1, t2, "BO3", df_stats, modelo, n_sims)
    perd1 = t2 if venc1 == t1 else t1
    print(f"   {t1} vs {t2}  --> Vencedor: {venc1} (Placar: {plac1})")

    venc2, plac2 = rodar_monte_carlo(t3, t4, "BO3", df_stats, modelo, n_sims)
    perd2 = t4 if venc2 == t3 else t3
    print(f"   {t3} vs {t4}  --> Vencedor: {venc2} (Placar: {plac2})\n")

    # UPPER BRACKET FINAL
    print("🌟 FINAL DA CHAVE SUPERIOR (Melhor de 3)")
    venc_up, plac_up = rodar_monte_carlo(venc1, venc2, "BO3", df_stats, modelo, n_sims)
    perd_up = venc2 if venc_up == venc1 else venc1
    print(f"   {venc1} vs {venc2} --> Vai para a Grande Final: {venc_up} (Placar: {plac_up})\n")

    # LOWER BRACKET ROUND 1
    print("💀 PARTIDA ELIMINATÓRIA (Melhor de 3)")
    sobrev, plac_elim = rodar_monte_carlo(perd1, perd2, "BO3", df_stats, modelo, n_sims)
    eliminado1 = perd2 if sobrev == perd1 else perd1
    print(f"   {perd1} vs {perd2} --> Sobrevive: {sobrev} | ELIMINADO: {eliminado1} ❌ (Placar: {plac_elim})\n")

    # LOWER BRACKET FINAL
    print("🔥 FINAL DA CHAVE INFERIOR (Melhor de 3)")
    venc_low, plac_low = rodar_monte_carlo(perd_up, sobrev, "BO3", df_stats, modelo, n_sims)
    eliminado2 = sobrev if venc_low == perd_up else perd_up
    print(f"   {perd_up} vs {sobrev} --> Vai para a Grande Final: {venc_low} | ELIMINADO: {eliminado2} ❌ (Placar: {plac_low})\n")

    # GRANDE FINAL
    print("👑 GRANDE FINAL (Melhor de 5)")
    campeao, plac_final = rodar_monte_carlo(venc_up, venc_low, "BO5", df_stats, modelo, n_sims)
    vice = venc_low if campeao == venc_up else venc_up
    print(f"   {venc_up} vs {venc_low} --> CAMPEÃO DO GRUPO: {campeao} 🏆 (Placar: {plac_final})\n")

    print("==================================================")
    print(f"✈️ EQUIPE CLASSIFICADA PARA O SWISS STAGE: [{campeao}]")
    print("==================================================")

# 3. EXECUTAR A SIMULAÇÃO!
simular_grupo_playin_uma_vaga("Cloud9", "Karmine Corp", "FURIA", "MVK Esports", df_features, modelo_lol, 1000)

In [22]:
#CÉLULA 10
def simular_serie(time_a, time_b, tipo_serie, df_stats, modelo, time_a_blue_jogo1=True):
    vitorias = {time_a: 0, time_b: 0}

    if tipo_serie == 'BO1': vitorias_necessarias = 1
    elif tipo_serie == 'BO3': vitorias_necessarias = 2
    elif tipo_serie == 'BO5': vitorias_necessarias = 3
    else: return None

    # Jogo 1
    time_azul = time_a if time_a_blue_jogo1 else time_b
    time_vermelho = time_b if time_a_blue_jogo1 else time_a

    while vitorias[time_a] < vitorias_necessarias and vitorias[time_b] < vitorias_necessarias:
        vencedor_mapa = simular_mapa_mc(time_azul, time_vermelho, df_stats, modelo)

        if not vencedor_mapa:
            return None # Tratamento de erro se faltar dados

        vitorias[vencedor_mapa] += 1

        # Interrompe se alguém já atingiu os pontos necessários
        if vitorias[time_a] == vitorias_necessarias or vitorias[time_b] == vitorias_necessarias:
            break

        # Regra do competitivo: Perdedor escolhe o lado. Assumimos escolha do Blue Side.
        perdedor_mapa = time_a if vencedor_mapa == time_b else time_b
        time_azul = perdedor_mapa
        time_vermelho = time_b if perdedor_mapa == time_a else time_a

    vencedor_serie = time_a if vitorias[time_a] > vitorias[time_b] else time_b
    return vencedor_serie, vitorias

In [ ]:
#CÉLULA 11
def rodar_monte_carlo(time_a, time_b, tipo_serie, df_stats, modelo, n_simulacoes=1000):
    vitorias_totais = {time_a: 0, time_b: 0}
    placar_mais_comum = {}

    for i in range(n_simulacoes):
        # Para a Fase Suíça, podemos alternar aleatoriamente quem começa no Blue Side no Jogo 1
        time_a_blue_j1 = random.choice([True, False])

        resultado = simular_serie(time_a, time_b, tipo_serie, df_stats, modelo, time_a_blue_j1)

        if not resultado:
            return "Erro: Equipas não encontradas no histórico."

        vencedor_serie, placar = resultado
        vitorias_totais[vencedor_serie] += 1

        # Regista o placar (ex: "2-1") para análises aprofundadas
        placar_str = f"{placar[time_a]}-{placar[time_b]}"
        placar_mais_comum[placar_str] = placar_mais_comum.get(placar_str, 0) + 1

    prob_a = (vitorias_totais[time_a] / n_simulacoes) * 100
    prob_b = (vitorias_totais[time_b] / n_simulacoes) * 100

    # Encontra o placar mais provável
    placar_provavel = max(placar_mais_comum, key=placar_mais_comum.get)

    return {
        'Confronto': f"{time_a} vs {time_b}",
        'Formato': tipo_serie,
        f'Chance {time_a} (%)': round(prob_a, 2),
        f'Chance {time_b} (%)': round(prob_b, 2),
        'Placar Mais Provável': placar_provavel
    }

# Exemplo de teste rápido com as equipas que você simulou anteriormente:
resultado_mc = rodar_monte_carlo("Cloud9", "Karmine Corp", "BO3", df_features, modelo_lol, n_simulacoes=2000)
resultado_mc = rodar_monte_carlo("Cloud9", "Karmine Corp", "BO3", df_features, modelo_lol, n_simulacoes=2000)
print(resultado_mc)